In [11]:
import sys
import os
from pydataxm.pydatasimem import ReadSIMEM, CatalogSIMEM
import pandas as pd

def get_df(id, fecha_inicio, fecha_final, nombre_csv="archivo.csv"):
    # --- Obtención de datos ---
    catalogo = CatalogSIMEM(catalog_type='Datasets')
    df_catalogo = catalogo.get_data()

    dataset_id = id
    fecha_fin = fecha_final
    simem = ReadSIMEM(dataset_id, fecha_inicio, fecha_fin)
    df_general = simem.main()

    # --- Calcular ruta DOS niveles arriba, sin crear carpetas ---
    # Si __file__ no existe (Jupyter/REPL), usamos el cwd
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()  # entorno interactivo

    carpeta_dos_arriba = os.path.abspath(os.path.join(base_dir, ".."))

    # Verificación: no crear carpetas; si no existe, fallar con mensaje claro
    if not os.path.isdir(carpeta_dos_arriba):
        raise FileNotFoundError(
            f"La carpeta dos niveles arriba no existe o no es accesible: {carpeta_dos_arriba}"
        )

    ruta_csv = os.path.join(carpeta_dos_arriba, nombre_csv)

    # Manejo defensivo si la librería devuelve None
    if df_general is None:
        df_general = pd.DataFrame()
        print("Advertencia: ReadSIMEM.main() devolvió None. Se guarda DataFrame vacío.")

    # Guardar CSV (no se crean carpetas)
    df_general.to_csv(ruta_csv, index=False)
    print(f"CSV guardado en: {ruta_csv}")

    return df_general

In [12]:
import requests
from typing import List, Any, Dict, Union

def obtener_namecolumns(dataset_id: str, url_template: str, timeout: int = 200) -> List[str]:
    
    # 1) Construir la URL reemplazando el placeholder exactamente como lo tienes en tu código
    if "{dataset_id}" not in url_template:
        raise ValueError("El url_template debe contener el placeholder {dataset_id}")
    url = url_template.format(dataset_id=dataset_id)

    # 2) Llamar al endpoint
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()

    # 3) Parsear JSON
    try:
        data = resp.json()
    except ValueError as e:
        raise ValueError(f"La respuesta no es JSON válido. Error: {e}")

    if data is None:
        raise ValueError("La respuesta JSON está vacía (None).")

    # 4) Buscar recursivamente 'Columns' y extraer 'nameColumn'
    def _find_namecolumns(obj: Union[Dict[str, Any], List[Any]]) -> List[str]:
        found: List[str] = []
        columns_keys_lower = {"columns"}  # case-insensitive ('Columns' o 'columns')

        def _walk(node: Any):
            if isinstance(node, dict):
                for k, v in node.items():
                    # ¿Esta clave es 'Columns' (sin importar mayúsculas)?
                    if str(k).lower() in columns_keys_lower:
                        # v puede ser list o dict. Extraer 'nameColumn' en formatos comunes.
                        if isinstance(v, list):
                            for item in v:
                                if isinstance(item, dict) and "nameColumn" in item:
                                    found.append(item["nameColumn"])
                        elif isinstance(v, dict):
                            # Buscar en subclaves típicas
                            candidates_list = None
                            for ck in ("items", "data", "list", "values"):
                                if ck in v and isinstance(v[ck], list):
                                    candidates_list = v[ck]
                                    break
                            if candidates_list:
                                for item in candidates_list:
                                    if isinstance(item, dict) and "nameColumn" in item:
                                        found.append(item["nameColumn"])
                            else:
                                # Último recurso: barrer el dict y listas internas
                                for subv in v.values():
                                    if isinstance(subv, dict) and "nameColumn" in subv:
                                        found.append(subv["nameColumn"])
                                    elif isinstance(subv, list):
                                        for item in subv:
                                            if isinstance(item, dict) and "nameColumn" in item:
                                                found.append(item["nameColumn"])
                    # Seguir recorriendo el árbol
                    _walk(v)
            elif isinstance(node, list):
                for it in node:
                    _walk(it)

        _walk(obj)

        # Devolver únicos conservando el orden
        seen = set()
        unique = []
        for x in found:
            if x not in seen:
                unique.append(x)
                seen.add(x)
        return unique

    return _find_namecolumns(data)


##EJEMPLO DE USO###
#dataset_id = "75f675"
#url_template = "https://www.simem.co/backend-files/api/detalle-datos-publicos?datasetId={dataset_id}"


#columnas = obtener_namecolumns(dataset_id, url_template)
#print("nameColumn encontrados:", columnas)

In [14]:
def Separacion(columnas, archivo):
        
    while True:
        var_x = input("Variable eje X: ").strip()
        if var_x in columnas:
            break

    while True:
        var_y = input("Variable eje y: ").strip()
        if var_y in columnas:
            break

    df_general = archivo[[var_x, var_y]].copy()
    

    try:
        # Tu formato “ideal”
        df_general[var_x] = pd.to_datetime(df_general[var_x], format='%Y-%m-%d %H:%M:%S')
    except Exception:
        # Fallback flexible si hay variaciones
        df_general[var_x] = pd.to_datetime(df_general[var_x], errors='coerce', dayfirst=False)

    df_atributos = archivo.drop([var_x, var_y], axis=1)

    return df_general, df_atributos

In [16]:
id = "055A4D"
inicio = "2025-01-01"
fin = "2025-01-01"
url = "https://www.simem.co/backend-files/api/detalle-datos-publicos?datasetId={dataset_id}"

print("Etapa1")
get_df(id,inicio,fin)
print("Etapa2")
colums = obtener_namecolumns(id, url)
print(colums)
print("Etapa3")
df = pd.read_csv("../archivo.csv")
print('Etapa4')
df1 , df2 = Separacion(colums,df)
print(df1.head())
print(df2.head())

Etapa1
****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Generación real"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0011227130889892578
Extraccion de registros: 8.407366037368774
End of data extracting process
****************************************************************************************************
CSV guardado en: d:\ambiente\escritorio\COSITAS_DE_PY\ProyectoPracticas\archivo.csv
Etapa2
['CodigoVariable', 'Valor', 'CodigoPlanta', 'UnidadMedida', 'CodigoSICAgente', 'Version', 'FechaHora', 'CodigoDuracion']
Etapa3
Etapa4


: 

: 

In [8]:
from pydataxm.pydatasimem import CatalogSIMEM

catalog_datasets = CatalogSIMEM(catalog_type='Datasets')

df_catalog_datasets = catalog_datasets.get_data()

df_catalog_datasets.head()

,idDataset,nombreConjuntoDatos,fechaPublicacion,fechaActualizacion,inicioDato,finDato,fechaDescarga,urlConexionAPI,urlConjuntoDatos,tipoPublicacion
0,a704ee,Datos soporte del proceso de liquidación por C...,2023-09-28T21:28:06.433,2026-02-11T12:32:35.917,2017-12-01T00:00:00,2025-12-01T00:00:00,2025-04-30T11:38:47.52,https://www.simem.coPublicData?startDate=2026-...,https://www.simem.co/datadetail/a704eef3-ca1c-...,Público
1,d808d4,Energía de Referencia para el Mercado Secundario,2023-09-29T11:01:54.317,2025-09-03T14:56:00.327,2018-12-01T00:00:00,2050-11-30T00:00:00,2025-04-09T09:42:23.763,https://www.simem.coPublicData?startDate=2026-...,https://www.simem.co/datadetail/d808d43d-e3da-...,Público
2,2106b8,Precios y estadísticas de los contratos de con...,2023-09-30T10:35:01.19,2026-02-11T13:29:27.423,2020-10-01T00:00:00,2026-01-01T00:00:00,2025-03-08T11:37:23.957,https://www.simem.coPublicData?startDate=2026-...,https://www.simem.co/datadetail/99686983-57a4-...,Público
3,B1009C,Compras en bolsa nacional en moneda,2024-11-07T10:33:56.017,2026-02-19T10:26:22.827,2021-01-01T00:00:00,2026-02-14T00:00:00,NaN,https://www.simem.coPublicData?startDate=2026-...,https://www.simem.co/datadetail/b1009c18-5450-...,Público
4,E4CE10,Costo Servicios ASIC y CND,2025-07-11T11:30:08.153,2026-02-11T12:25:44.95,2021-01-01T00:00:00,2026-01-01T00:00:00,NaN,https://www.simem.coPublicData?startDate=2026-...,https://www.simem.co/datadetail/e4ce10c3-dffc-...,Público


In [4]:
df_data = pd.read_csv("../archivo.csv", index_col= False)
df_data.head(5)


,CodigoVariable,FechaHora,CodigoDuracion,UnidadMedida,CodigoSICAgente,CodigoPlanta,Version,Valor
0,GReal,2025-01-01 23:00:00,PT1H,kWh,EEPG,2UP2,TXR,0.0
1,GReal,2025-01-01 22:00:00,PT1H,kWh,EEPG,2UP2,TXR,0.0
2,GReal,2025-01-01 21:00:00,PT1H,kWh,EEPG,2UP2,TXR,0.0
3,GReal,2025-01-01 20:00:00,PT1H,kWh,EEPG,2UP2,TXR,0.0
4,GReal,2025-01-01 19:00:00,PT1H,kWh,EEPG,2UP2,TXR,0.0


In [5]:
df_data.index

RangeIndex(start=0, stop=53376, step=1)

In [ ]:
def plot_df(df):
    df = df.copy()
    df['FechaHora'] = pd.to_datetime(df['FechaHora'], dayfirst=True, errors='coerce')
    df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
    df = df.dropna(subset=['FechaHora', 'Valor']).sort_values('FechaHora')
    plt.figure(figsize=(10,4))
    plt.plot(df['FechaHora'], df['Valor'], marker='o', linestyle='-')
    plt.xlabel('FechaHora')
    plt.ylabel('Valor')
    plt.title('Serie temporal — Valor vs FechaHora')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_df(df_xy_principal)